# Libraries

In [1]:
import os
import shutil

import pandas as pd
import geopandas as gpd
import rasterio as rio
import pickle
import osmnx as ox
import networkx as nx
import numpy as np
import math

from pyproj import Transformer
from tqdm import tqdm

import warnings
warnings.filterwarnings("ignore")

In [2]:
# Set home directory
home_dir = '/Users/yiyi/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology/Research/Global road network resilience/01_data' # CURA
home_dir = '/Users/yiyi/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology(2)/Research/Global road network resilience/01_data' # SCARP
# home_dir = '/Users/yiyi/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology/Research/Global road network resilience/01_data' # Home

In [60]:
'''Housekeeping items'''
# Make a copy of networks that are affected by hurricane CAT1-5
coastal_networkID_df = pd.read_csv(home_dir + '/Coastal_flooding/network_polys_IDs.csv')

# Array of network ids
netids = coastal_networkID_df.net_id.values.astype(int)

# Copy graphs
source_folder = home_dir + '/all_graph_flooded/'
target_folder = home_dir + '/Coastal_flooding/graphs/'

# Find networks in netids that are affected by coastal flooding and make a copy to new location
for file in os.listdir(source_folder):
    if file.endswith('.pk'):
        netid = int(file.split('_')[4])
        if netid in netids:
            # make a copy
            shutil.copy2(os.path.join(home_dir,source_folder, file),
                         os.path.join(home_dir,target_folder, file))
        else:
            continue

# Attach inundation values

In [134]:
# Location of graphs
graphs_dir = home_dir + '/Coastal_flooding/graphs/'

# Location of coastal flood risk maps
coastal_flooding_dir = '/Users/yiyi/Desktop/Coastal Flooding/NOAA_National_storm_surge/US_SLOSH_MOM_Inundation_20250923'

hurricane_categories = [1,2,3,4,5]

hurricane_rasters = {}
hurricane_bands = {}

for hurricane_category in hurricane_categories:
    # coastal flood risk map raster
    src = rio.open(f'/Users/yiyi/Desktop/Coastal Flooding/NOAA_National_storm_surge/US_SLOSH_MOM_Inundation_20250923/us_Category{hurricane_category}_MOM_Inundation_HIGH.tif')

    hurricane_rasters[hurricane_category] = src
    hurricane_bands[hurricane_category] = src.read(1) # load raster band
# Transform to lat lon
transformer = Transformer.from_crs(
    "EPSG:4326",      # lon/lat
    src.crs,  # EPSG:4269
    always_xy=True
)

**Dictionary for NOAA Hurricane risk maps**  
Value: description  
1: 00 to 01 foot above ground<br>
2: 01 to 02 feet above ground<br>
3: 02 to 03 feet above ground<br>
...<br>
20: 19 to 20 feet above ground<br>
21: Greater than 20 feet above ground<br>
99: Levee Areas - Consult Local Officials for flood risk<br>

In [142]:
for file in tqdm(os.listdir(graphs_dir)):
    if file.endswith('.pk'):
        # Read in the graph pickle file
        g = pickle.load(open(graphs_dir + file, 'rb'))
        # Network id
        net_id = int(file.split("_")[4])

        #
        for node, data in g.nodes(data=True):
            lon = data["x"]
            lat = data["y"]

            # Reproject lon/lat to raster crs
            x, y = transformer.transform(lon, lat)

            # Iterate through hurricane categories
            for hurricane_category in hurricane_categories:
                # Load hurricane raster, band
                src = hurricane_rasters[hurricane_category]
                band = hurricane_bands[hurricane_category]
                # Get raster value at node
                try:
                    row, col = src.index(x, y)
                    value = band[row, col]

                    if nodata is not None and value == nodata:
                        value = np.nan

                except IndexError:
                    value = np.nan  # node outside raster extent
                # Attach raster value to node attribute
                g.nodes[node][f"CAT_{hurricane_category}_value"] = value

        # Save the graph
        with open(home_dir + f"/Coastal_flooding/graphs_node_flooded/net_{net_id}.pk", "wb") as f:
            pickle.dump(g, f, protocol=2)

100%|██████████| 56/56 [03:11<00:00,  3.43s/it]


In [182]:
# Add inundation values on graph edges
def take_val(i, j):
    if math.isnan(i) and math.isnan(j):
        return float("nan")
    if math.isnan(i):
        return j
    if math.isnan(j):
        return i
    return max(i, j)

graph_flood_dir = home_dir + '/Coastal_flooding/graphs_node_flooded/'
for file in tqdm(os.listdir(graph_flood_dir)):
    if file.endswith('.pk'):
        net_id = int("".join(filter(str.isdigit, file)))
        # Read in the graph pickle file
        g = pickle.load(open(graph_flood_dir + file, 'rb'))
        for i,j,data in g.edges.data():
            for hurricane_category in hurricane_categories:
                i_val = g.nodes[i][f'CAT_{hurricane_category}_value']
                j_val = g.nodes[j][f'CAT_{hurricane_category}_value']
                val = take_val(i_val,j_val)
                g.edges[i, j, 0][f'CAT_{hurricane_category}_value'] = val
        # Save the graph
        with open(home_dir + f"/Coastal_flooding/graphs_node_edge_flooded/net_{net_id}.pk", "wb") as f:
            pickle.dump(g, f, protocol=2)

# Direct exposure

In [244]:
hurricane_categories = [1,2,3,4,5]

net_id_col = []
total_length_col = []
cat1_expo_col = []
cat2_expo_col = []
cat3_expo_col = []
cat4_expo_col = []
cat5_expo_col = []
cat_lsts = [cat1_expo_col, cat2_expo_col, cat3_expo_col, cat4_expo_col, cat5_expo_col]
# 
for file in tqdm(os.listdir(home_dir + "/Coastal_flooding/graphs_node_edge_flooded/")):

    net_id = int("".join(filter(str.isdigit, file)))
    net_id_col.append(net_id)
    g = pickle.load(open(home_dir + "/Coastal_flooding/graphs_node_edge_flooded/"+ file, 'rb'))
    
    total_length = 0
    for i,j,data in g.edges.data():
        total_length += data["length"]
    total_length_col.append(total_length)

    for hurricane_category in hurricane_categories:
        
        exposure_length = 0
        
        for i,j,data in g.edges.data():

            try:

                if math.isnan(data[f"CAT_{hurricane_category}_value"]):
                    continue
                elif data[f"CAT_{hurricane_category}_value"]>1:
                    exposure_length += data["length"]
            except KeyError:
                continue

        cat_lsts[hurricane_category-1].append(exposure_length)
    
exposure_data = {
    "net_id": net_id_col,
    "total_length" : total_length_col,
    "CAT1_expo" : cat1_expo_col,
    "CAT2_expo" : cat2_expo_col,
    "CAT3_expo" : cat3_expo_col,
    "CAT4_expo" : cat4_expo_col,
    "CAT5_expo" : cat5_expo_col
}

exposure_data_df = pd.DataFrame.from_dict(exposure_data)

exposure_data_df.head(3)

# Calculate exposure percentage
for hurricane_category in hurricane_categories:
    exposure_data_df[f'CAT{hurricane_category}_expo_perc'] = 100*exposure_data_df[f'CAT{hurricane_category}_expo']/exposure_data_df['total_length']
    
exposure_data_df.to_csv(home_dir + "/Coastal_flooding/graph_exposure.csv")

100%|██████████| 55/55 [01:00<00:00,  1.09s/it]


# Indirect impact

Original OD pairs

In [ ]:
# Move original dry wet simulation results files to new location (US coastal networks only)
exposure_df = pd.read_csv(home_dir + '/Coastal_flooding/graph_exposure.csv', index_col=0)
net_ids = list(exposure_df.net_id)

for file in os.listdir(home_dir + '/OD_routing_simulation_results/2576_dry_wet_OD_routing_30cm/'):
    if file.endswith('.csv'):
        net_id = int("".join(filter(str.isdigit, file)))
        if net_id in net_ids:
            # Copy the file to new location
            shutil.copy(home_dir + '/OD_routing_simulation_results/2576_dry_wet_OD_routing_30cm/' + file,
                        home_dir + '/Coastal_flooding/Dry_wet_routing_30cm/')

Create disrupted graphs with flood travel speed and travel time

In [96]:
# Generate disrupted graphs with adjusted travel time and travel speed
hurricane_categories = [1,2,3,4,5]

for graph_file in tqdm(os.listdir(home_dir + '/Coastal_flooding/graphs_node_edge_flooded/')):
    if graph_file.endswith('.pk'):
        g = pickle.load(open(home_dir + "/Coastal_flooding/graphs_node_edge_flooded/"+ graph_file, 'rb'))
        net_id = int("".join(filter(str.isdigit, graph_file)))
        if net_id == 2091:
            continue
        else:
            # Iterate through hurricane categories [1,2,3,4,5]
            for cat in hurricane_categories:
                # Remove edges with 1 foot inundation
                g_disrupted = g.copy() # initiate the disrupted graph
                for i,j,data in g.edges.data():
                    try:
                        if math.isnan(data[f"CAT_{cat}_value"]):  # if there is less than 1 foot inundation
                            continue
                        else:
                            # remove edge since there is more than 1 foot flood inundation
                            g_disrupted.remove_edge(i,j)
                    except KeyError:
                        continue
                # calculate normal/baseline travel time and travel speed
                g_disrupted = ox.routing.add_edge_speeds(g_disrupted)
                g_disrupted = ox.routing.add_edge_travel_times(g_disrupted)

                # Add flood travel speed and travel time
                # Travel speed as a function of travel time:
                # v = 0.0009w^2 - 0.5529w + 86.9448 (Pregnolato et al 2017)
                # v: km/h;   w:mm
                # When w is 1 foot or 304.8mm, v= 2.03 km/h which is 0.564 m/s
                for i,j,data in g_disrupted.edges.data():
                    # baseline travel speed
                    baseline_speed_kph = data['speed_kph']
                    baseline_speed_ms = baseline_speed_kph/3.6
                    flood_speed_ms = 2.03
                    # Calculate flood speed (m/s) and time (s)
                    cat_speed_ms = min(flood_speed_ms, baseline_speed_ms)
                    cat_time_s = data['length']/cat_speed_ms
                    # Add flood speed and time
                    g_disrupted[i][j][0][f'CAT{cat}_speed_ms'] = cat_speed_ms
                    g_disrupted[i][j][0][f'CAT{cat}_time_s'] = cat_time_s
                
                # Save disrupted graph
                with open(home_dir + f"/Coastal_flooding/graphs_disrupted/CAT{cat}/net_{net_id}_disrupted_speed_time.pk", "wb") as f:
                    pickle.dump(g_disrupted, f, protocol=2)

100%|██████████| 55/55 [13:10<00:00, 14.38s/it]


Process network id 2091

In [97]:
net_id = 2091
g = pickle.load(open(home_dir + f"/Coastal_flooding/graphs_node_edge_flooded/net_{net_id}.pk", 'rb'))
# Iterate through hurricane categories [1,2,3,4,5]
for cat in hurricane_categories:
    # Remove edges with 1 foot inundation
    g_disrupted = g.copy() # initiate the disrupted graph
    for i,j,data in g.edges.data():
        try:
            if math.isnan(data[f"CAT_{cat}_value"]):  # if there is less than 1 foot inundation
                continue
            else:
                # remove edge since there is more than 1 foot flood inundation
                g_disrupted.remove_edge(i,j)
        except KeyError:
            continue
    # calculate normal/baseline travel time and travel speed

    hwy_speeds = { # km/h
    'motorway': 120,
    'motorway_link': 120,
    'trunk': 100,
    'trunk_link': 100,
    'primary': 90,
    'primary_link': 90,
    'secondary': 80,
    'secondary_link': 80,
    'tertiary': 60,
    'tertiary_link': 60}
    
    g_disrupted = ox.routing.add_edge_speeds(g_disrupted, hwy_speeds=hwy_speeds, fallback=50)
    g_disrupted = ox.routing.add_edge_travel_times(g_disrupted)

    # Add flood travel speed and travel time
    # Travel speed as a function of travel time:
    # v = 0.0009w^2 - 0.5529w + 86.9448 (Pregnolato et al 2017)
    # v: km/h;   w:mm
    # When w is 1 foot or 304.8mm, v= 2.03 km/h which is 0.564 m/s
    for i,j,data in g_disrupted.edges.data():
        # baseline travel speed
        baseline_speed_kph = data['speed_kph']
        baseline_speed_ms = baseline_speed_kph/3.6
        flood_speed_ms = 2.03
        # Calculate flood speed (m/s) and time (s)
        cat_speed_ms = min(flood_speed_ms, baseline_speed_ms)
        cat_time_s = data['length']/cat_speed_ms
        # Add flood speed and time
        g_disrupted[i][j][0][f'CAT{cat}_speed_ms'] = cat_speed_ms
        g_disrupted[i][j][0][f'CAT{cat}_time_s'] = cat_time_s
    
    # Save disrupted graph
    with open(home_dir + f"/Coastal_flooding/graphs_disrupted/CAT{cat}/net_{net_id}_disrupted_speed_time.pk", "wb") as f:
        pickle.dump(g_disrupted, f, protocol=2)

Add travel time and speed to normal networks

In [148]:
for file in os.listdir(home_dir + '/Coastal_flooding/Dry_wet_routing_30cm/'):
    if file.endswith('.csv'):
        net_id = int("".join(filter(str.isdigit, file)))
        #
        g_baseline = pickle.load(open(home_dir + f'/Coastal_flooding/graphs_node_edge_flooded/net_{net_id}.pk', 'rb'))
        g_baseline = ox.routing.add_edge_speeds(g_baseline)
        g_baseline = ox.routing.add_edge_travel_times(g_baseline)
        # Save disrupted graph
        with open(home_dir + f"/Coastal_flooding/graphs_speed_time/net_{net_id}_speed_time.pk", "wb") as f:
            pickle.dump(g_baseline, f, protocol=2)

Simulation on disrupted networks

In [ ]:
hurricane_categories = [1,2,3,4,5]

# Processed files
visited = []
for item in os.listdir(home_dir + f'/Coastal_flooding/OD_simulation_result/'):
    visited.append(int("".join(filter(str.isdigit, item))))

for file in tqdm(os.listdir(home_dir + '/Coastal_flooding/Dry_wet_routing_30cm/')):
    if file.endswith('.csv'):
        # Read the original dry wet simulation files
        og_dry_wet_df = pd.read_csv(home_dir + '/Coastal_flooding/Dry_wet_routing_30cm/' + file, index_col=0)
        net_id = int("".join(filter(str.isdigit, file)))

        if net_id in visited: # if visited
            continue

        else:

            # Iterate through 5 hurricane categories
            for cat in hurricane_categories:
                # Load the corresponding disrupted graph
                g_disrupted = pickle.load(open(home_dir + f"/Coastal_flooding/graphs_disrupted/CAT{cat}/net_{net_id}_disrupted_speed_time.pk", 'rb'))
                
                OD_route_cat_lst = []
                OD_length_cat_lst = []
                OD_time_cat_lst = []
                
                # iterate through OD pairs
                for row in range(og_dry_wet_df.shape[0]):
                    # Initialize
                    OD_route_cat = np.nan
                    OD_length_cat = np.nan
                    OD_time_cat = np.nan
                    # Get the ids of O and D
                    O_id = og_dry_wet_df.iloc[row]['O_node_id']
                    D_id = og_dry_wet_df.iloc[row]['D_node_id']
                    # Check if the both O and D are still in the disrupted graph
                    if O_id in g_disrupted.nodes and D_id in g_disrupted.nodes:
                        try:
                            # Find route between O and D
                            OD_route_cat = nx.shortest_path(g_disrupted, O_id, D_id, weight=f'CAT{cat}_time_s') # route

                            # Route travel distance and travel time
                            # Initialize totals
                            route_time = 0
                            route_length = 0
                            # Iterate over consecutive nodes in the route
                            for u, v in zip(OD_route_cat[:-1], OD_route_cat[1:]):
                                edge_data = g_disrupted.get_edge_data(u, v)
                                # Some edges may have multiple entries (MultiDiGraph)
                                # Take the first one
                                if isinstance(edge_data, dict) and 0 in edge_data:
                                    edge_attr = edge_data[0]
                                else:
                                    edge_attr = edge_data
                                    
                                route_time += edge_attr.get(f'CAT{cat}_time_s', 0)     # in seconds 
                                route_length += edge_attr.get('length', 0)             # in meters
                            
                        except nx.NetworkXNoPath:
                            OD_route_cat_lst.append(np.nan)
                            OD_length_cat_lst.append(np.nan)
                            OD_time_cat_lst.append(np.nan)
                            continue
                    
                    # Add result to list    
                    OD_route_cat_lst.append(OD_route_cat)
                    OD_length_cat_lst.append(route_length)
                    OD_time_cat_lst.append(route_time)
                    
                # Add simulation results as additional columns
                og_dry_wet_df[f'OD_route_CAT_{cat}'] = OD_route_cat_lst
                og_dry_wet_df[f'OD_length_CAT_{cat}'] = OD_length_cat_lst
                og_dry_wet_df[f'OD_time_CAT_{cat}'] = OD_time_cat_lst

            # Travel route, distance and time under baseline scenario
            # load graph
            g_baseline = pickle.load(open(home_dir + f'/Coastal_flooding/graphs_speed_time/net_{net_id}_speed_time.pk', 'rb'))

            OD_route_baseline_lst = []
            OD_length_baseline_lst = []
            OD_time_baseline_lst = []

            # iterate through OD pairs
            for row in range(og_dry_wet_df.shape[0]):
                # Initialize
                OD_route_baseline = np.nan
                OD_length_baseline = np.nan
                OD_time_baseline = np.nan
                # Get the ids of O and D
                O_id = og_dry_wet_df.iloc[row]['O_node_id']
                D_id = og_dry_wet_df.iloc[row]['D_node_id']

                OD_route_baseline = nx.shortest_path(g_baseline, O_id, D_id, weight='travel_time') # route
                # Route travel distance and travel time
                # Initialize totals
                route_time = 0
                route_length = 0
                # Iterate over consecutive nodes in the route
                for u, v in zip(OD_route_baseline[:-1], OD_route_baseline[1:]):
                    edge_data = g_baseline.get_edge_data(u, v)
                    # Some edges may have multiple entries (MultiDiGraph)
                    # Take the first one
                    if isinstance(edge_data, dict) and 0 in edge_data:
                        edge_attr = edge_data[0]
                    else:
                        edge_attr = edge_data
                        
                    route_time += edge_attr.get('travel_time', 0)     # in seconds 
                    route_length += edge_attr.get('length', 0)        # in meters
                
                OD_route_baseline_lst.append(OD_route_baseline)
                OD_length_baseline_lst.append(route_length)
                OD_time_baseline_lst.append(route_time)
            # Add simulation results as additional columns
            og_dry_wet_df['OD_route_baseline'] = OD_route_baseline_lst
            og_dry_wet_df['OD_length_baseline'] = OD_length_baseline_lst
            og_dry_wet_df['OD_time_baseline'] = OD_time_baseline_lst

            # Save file
            og_dry_wet_df.to_csv(home_dir + f'/Coastal_flooding/OD_simulation_result/G_{net_id}_cat_routing.csv')

 16%|█▋        | 9/55 [00:00<00:00, 76.15it/s]

In [160]:
# Processed files
visited = []
for item in os.listdir(home_dir + f'/Coastal_flooding/OD_simulation_result/'):
    visited.append(int("".join(filter(str.isdigit, item))))

In [161]:
visited

[2018, 1793, 1971, 2172, 2048, 2123, 2054, 2386, 2187, 2052, 2055]